<a href="https://colab.research.google.com/github/npmauad/dataeng/blob/main/notebooks/TP1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Trabajo Práctico N° 1**

*Declaración de uso de IA: Utilicé Claude Sonnet 5 Medium para ayuda en la escritura del código y resolución de dudas.*

**1. Importar los archivos correspondientes a eventos posteriores al 02-09-2026**

In [3]:
# Importar los paquetes necesarios

import requests
import io
import zipfile
import pandas as pd

# Traer la lista de archivos con get
url_to_get = "https://data.gdeltproject.org/gdeltv2/masterfilelist.txt"
masterfile = requests.get(url_to_get)
lines = masterfile.text.splitlines()

# Quedarnos solo con los archivos desde el 02-09-2026
datefrom = "20260902"

export_urls = []
for line in lines:
    parts = line.split(" ")
    file_url = parts[-1]  # formato: "tamaño hash URL"
    if file_url.endswith(".export.CSV.zip"):
        filedate = file_url.split("/")[-1][:8]  # primeros 8 dígitos del nombre
        if filedate >= datefrom:
            export_urls.append(file_url)

print(f"Archivos a descargar: {len(export_urls)}")

Archivos a descargar: 859


**2. Definir columnas. Descargar y leer cada archivo, y unificarlos en un solo DataFrame**

In [7]:
# Columnas del codebook GDELT 2.0 que necesitamos, con su posición
COLS_IDX = [0, 1, 32, 33, 59]
COLS_NAMES = ["GlobalEventID", "Day", "NumSources", "NumArticles", "DateAdded"]

# Función para leer cada archivo
def read_export_zip(file_url):
    r = requests.get(file_url)
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        nombre_csv = z.namelist()[0]
        with z.open(nombre_csv) as f:
            df = pd.read_csv(
                f,
                sep="\t",
                header=None,
                usecols=COLS_IDX,
                names=COLS_NAMES,
                dtype={"GlobalEventID": "int32", "Day": "int32",
                       "NumSources": "int16", "NumArticles": "int16"}
            )
    return df

# Descargar todo y unificar en un solo DataFrame
dfs = []
for i, u in enumerate(export_urls):
    dfs.append(read_export_zip(u))
    if i % 50 == 0:
        print(f"  {i}/{len(export_urls)} descargados")
df_events = pd.concat(dfs, ignore_index=True)
print(df_events.shape)

  0/859 descargados
  50/859 descargados
  100/859 descargados
  150/859 descargados
  200/859 descargados
  250/859 descargados
  300/859 descargados
  350/859 descargados
  400/859 descargados
  450/859 descargados
  500/859 descargados
  550/859 descargados
  600/859 descargados
  650/859 descargados
  700/859 descargados
  750/859 descargados
  800/859 descargados
  850/859 descargados
(904053, 5)


**3. Calcular las métricas diarias**

In [9]:
df_events["date"] = pd.to_datetime(df_events["DateAdded"].astype(str).str[:8], format="%Y%m%d") #Convertir las fechas de la columna DateAdded en formato de fecha

# Crear el resumen diario
daily_summary = df_events.groupby("date").agg(
    count_events=("GlobalEventID", "count"),
    avg_articles=("NumArticles", "mean"),
    avg_sources=("NumSources", "mean")
).reset_index()

# Mostrar el resumen diario
daily_summary

,date,count_events,avg_articles,avg_sources
0,2026-09-02,120105,4.642246,1.048508
1,2026-09-03,117020,4.630003,1.045454
2,2026-09-04,107037,4.686781,1.047012
3,2026-09-05,66878,4.640794,1.039923
4,2026-09-06,60454,4.574304,1.043835
5,2026-09-07,86043,4.677684,1.040259
6,2026-09-08,114917,4.601887,1.046956
7,2026-09-09,119065,4.625264,1.041893
8,2026-09-10,112534,4.606323,1.044138
